# LC4J vs Python-Native Frameworks: Benchmark Analysis

This notebook contains the statistical analysis plan as pre-registered for the study.

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 2. Helper Functions
Implementing Cliff's Delta and Bootstrap CIs for non-parametric evaluation.

In [ ]:
def cliffs_delta(x, y):
    """Calculate Cliff's delta effect size."""
    m, n = len(x), len(y)
    res = 0
    for i in x:
        for j in y:
            if i > j:
                res += 1
            elif i < j:
                res -= 1
    return res / (m * n)

def bootstrap_ci(data, stat_func=np.mean, alpha=0.05, n_boot=10000):
    """Percentile bootstrap confidence interval."""
    boot_stats = [stat_func(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lower = np.percentile(boot_stats, 100 * (alpha / 2))
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lower, upper

## 3. RQ1: Sustained Throughput Analysis
Given the paper's N=30 replicates, we simulate the distributions matching the reported empirical means and variance.

In [ ]:
# Simulated data based on empirical results (Table 3)
# Workload 1 (Single-turn) at concurrency 500
runs = 30
lc4j_w1 = np.random.normal(loc=4127, scale=95, size=runs)
lc_py_w1 = np.random.normal(loc=2244, scale=70, size=runs)
li_w1 = np.random.normal(loc=2117, scale=65, size=runs)
hs_w1 = np.random.normal(loc=2348, scale=75, size=runs)

df_w1 = pd.DataFrame({
    'Framework': ['LangChain4j']*runs + ['LangChain-Py']*runs + ['LlamaIndex']*runs + ['Haystack']*runs,
    'Throughput': np.concatenate([lc4j_w1, lc_py_w1, li_w1, hs_w1])
})

plt.figure(figsize=(10, 6))
sns.boxplot(x='Framework', y='Throughput', data=df_w1, palette='Set2')
plt.title('Throughput Distribution (W1, C=500)')
plt.ylabel('Requests per Second')
plt.show()

In [ ]:
# Mann-Whitney U and Cliff's Delta against best Python (Haystack for W1)
stat, p_val = stats.mannwhitneyu(lc4j_w1, hs_w1, alternative='greater')
delta = cliffs_delta(lc4j_w1, hs_w1)

print(f"Mann-Whitney U: {stat}, p-value: {p_val:.2e}")
print(f"Cliff's Delta: {delta:.2f} (Large effect size > 0.474)")

## 4. RQ2: Latency Tail Behavior
Analyzing the p99 orchestration latency as described in Section 7.3.

In [ ]:
# Table 4: W2 RAG p99 Orchestration Latency (ms)
lc4j_w2_p99 = np.random.lognormal(mean=np.log(24.71), sigma=0.1, size=runs)
lc_py_w2_p99 = np.random.lognormal(mean=np.log(78.4), sigma=0.2, size=runs)

df_lat = pd.DataFrame({
    'Framework': ['LangChain4j']*runs + ['LangChain-Py']*runs,
    'p99_Latency': np.concatenate([lc4j_w2_p99, lc_py_w2_p99])
})

plt.figure(figsize=(8, 5))
sns.kdeplot(data=df_lat, x='p99_Latency', hue='Framework', fill=True, common_norm=False)
plt.title('p99 Orchestration Latency Distribution (W2 RAG)')
plt.xlabel('Latency (ms)')
plt.show()

## 5. RQ3: Memory Footprint
Comparative analysis of process vs collective working set size (W_ss).

In [ ]:
# Table 5 RSS (MB)
mem_data = {
    'Workload': ['W1', 'W2', 'W3', 'W4'],
    'LC4J (1 JVM)': [1184, 1647, 1432, 1208],
    'Python Total (4 Workers)': [1872, 4012, 2736, 1916]
}
df_mem = pd.DataFrame(mem_data)
df_mem.set_index('Workload').plot(kind='bar', figsize=(10, 6), color=['#2ca02c', '#1f77b4'])
plt.title('Memory Footprint at Steady State')
plt.ylabel('Resident Set Size (MB)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()